In [6]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog("default", **{
    "type": "rest",
    "uri": "http://localhost:8181",
    "s3.endpoint": "http://localhost:9000",
    "s3.access-key-id": "minioadmin",
    "s3.secret-access-key": "minioadmin",
    "s3.path-style-access": "true",
})

print("Namespaces:")
print(catalog.list_namespaces())

print("\nBronze tables:")
print(catalog.list_tables("bronze"))

print("\nDefault tables:")
print(catalog.list_tables("default"))

Namespaces:
[('bronze',)]

Bronze tables:
[('bronze', 'who_gho_raw')]

Default tables:


NoSuchNamespaceError: NoSuchNamespaceException: Namespace does not exist: default

Cell 1 — check all tables exist with row counts

In [5]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog("default", **{
    "type": "rest",
    "uri": "http://localhost:8181",
    "s3.endpoint": "http://localhost:9000",
    "s3.access-key-id": "minioadmin",
    "s3.secret-access-key": "minioadmin",
    "s3.path-style-access": "true",
})

tables = [
    "bronze.who_gho_raw",
    "bronze.cdc_nndss_raw",
    "bronze.owid_covid_raw",
    "bronze.owid_vaccination_raw",
    "bronze.fred_macro_raw",
    "bronze.fda_adverse_events_raw",
]

for t in tables:

    namespace, table = t.split(".", 1)

    identifier = (namespace, table)

    exists = catalog.table_exists(identifier)

    if exists:
        rows = catalog.load_table(identifier).scan().to_arrow().num_rows

        print(f"{t}: {rows:,} rows")

    else:
        print(f"MISSING: {t}")

bronze.who_gho_raw: 61,555 rows
MISSING: bronze.cdc_nndss_raw
MISSING: bronze.owid_covid_raw
MISSING: bronze.owid_vaccination_raw
MISSING: bronze.fred_macro_raw
MISSING: bronze.fda_adverse_events_raw


Cell 2 — confirm lineage columns on every table

In [3]:
for t in tables:
    if not catalog.table_exists(t):
        continue
    df = catalog.load_table(t).scan(limit=1).to_pandas()
    has_ingested_at = "_ingested_at" in df.columns
    has_source_url  = "_source_url" in df.columns
    has_date        = "_ingestion_date" in df.columns
    status = "OK" if all([has_ingested_at, has_source_url, has_date]) else "MISSING COLUMNS"
    print(f"  {t}: {status}")

NameError: name 'tables' is not defined

Cell 3 — confirm Prefect flow run history

In [2]:
from prefect.client.orchestration import get_client

with get_client(sync_client=True) as client:
    flows = client.read_flow_runs(limit=5)

print(flows)

[FlowRun(id=UUID('89b730f7-9027-4cdd-9a1d-674c483e1b13'), name='purple-tody', flow_id=UUID('671cb84d-70db-4725-8e00-bf28b397df39'), state_id=UUID('019e4643-1e0b-7d04-a77a-33503220a94e'), deployment_id=None, deployment_version=None, work_queue_name=None, flow_version='97fb58362cceb396bb700c413f6646fc', parameters={}, idempotency_key=None, context={}, empirical_policy=FlowRunPolicy(max_retries=0, retry_delay_seconds=0.0, retries=0, retry_delay=0, pause_keys=set(), resuming=False, retry_type=None), tags=[], labels={'prefect.flow.id': '671cb84d-70db-4725-8e00-bf28b397df39'}, parent_task_run_id=None, run_count=1, expected_start_time=datetime.datetime(2026, 5, 20, 16, 35, 16, 514278, tzinfo=TzInfo(0)), next_scheduled_start_time=None, start_time=datetime.datetime(2026, 5, 20, 16, 35, 16, 641292, tzinfo=TzInfo(0)), end_time=datetime.datetime(2026, 5, 20, 16, 41, 4, 267931, tzinfo=TzInfo(0)), total_run_time=datetime.timedelta(seconds=347, microseconds=626639), estimated_run_time=datetime.timede